# Labeler 후보 평가 — 3모델 × 100샘플 Haiku 일치율 (Kaggle T4×2)

Qwen3-14B → Qwen3-30B-A3B → EXAONE-4.0-32B 를 **차례로** 돌려, `_audited3.jsonl`(Haiku 3-way 정답)과
**100건 일치율**을 비교한다. 가장 높은(≥90%) 모델을 c3 labeler(Haiku 대체)로 채택.
- transformers + 4bit(bitsandbytes) + device_map=auto → 세 모델 T4×2에 균일 로드, 모델 간 메모리 해제.
- 출력: 모델별 전체/클래스별 일치율 비교표.

## 1 clone + 설치

In [ ]:
import os
%cd /kaggle/working
if not os.path.exists('Calenda'):
    !git clone https://github.com/sooryong/Calenda.git
else:
    !cd Calenda && git fetch origin && git reset --hard origin/main
%cd /kaggle/working/Calenda
!git log --oneline -1
!pip -q install -U transformers accelerate bitsandbytes 2>/dev/null
import torch; print('GPU', torch.cuda.device_count(), torch.cuda.get_device_name(0))

## 2 criterion + 100 샘플(클래스 균형) 준비

> ⚠ **디스크**: 4bit는 full fp16를 받아 양자화 → 32B는 다운로드 ~64GB. Kaggle 디스크 부족 시: 모델 간 캐시 정리(루프에 포함됨)로 1개씩 처리하거나, 에 **AWQ 버전**(예: ) 사용, 또는 모델별 세션 분리.

In [ ]:
import json, sys, random
from pathlib import Path
sys.path.insert(0, 'scripts')
from _common import build_user_block
CRIT = Path('prompts/schedule_criterion.md').read_text(encoding='utf-8')
SYSTEM = ('너는 일정 분류기다. 아래 [기준]을 유일한 절대 기준으로 메시지의 has_schedule을 '
          'yes/pending/no 중 하나로 분류한다. JSON 하나만 출력: '
          '{"has_schedule":"yes|pending|no","reason":"한 줄"}. '
          '날짜·시각 있어도 거래/통보면 no. 거래 안내=no, 행사/모집 안내=pending.\n\n[기준]\n' + CRIT)

rows = [json.loads(l) for l in open('data/processed/_audited3.jsonl', encoding='utf-8') if l.strip()]
rows = [r for r in rows if not r.get('_audit_flag') and r['gold'].get('has_schedule') in ('yes','pending','no')]
by = {'yes':[], 'pending':[], 'no':[]}
for r in rows: by[r['gold']['has_schedule']].append(r)
random.seed(42)
sample = []
for k, n in (('yes',34), ('pending',33), ('no',33)):
    random.shuffle(by[k]); sample += by[k][:n]
GOLD = [r['gold']['has_schedule'] for r in sample]
PROMPTS_MSGS = [[{'role':'system','content':SYSTEM},
                 {'role':'user','content':build_user_block({'channel':r.get('channel',''),'received_at':r.get('received_at',''),
                   'sender':r.get('sender',''),'message':r.get('message',''),'thread_context':r.get('thread_context')})}]
                for r in sample]
from collections import Counter
print('샘플', len(sample), dict(Counter(GOLD)))

## 3 라벨 함수 + 모델 평가 루프

In [ ]:
import re, gc, json, torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)

def parse(text):
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.S)   # thinking 제거
    m = re.search(r'\{.*?\}', text, re.S)
    if not m: return None
    try:
        lab = (json.loads(m.group(0)).get('has_schedule') or '').strip().lower()
        return lab if lab in ('yes','pending','no') else None
    except Exception: return None

def evaluate(model_id):
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=BNB,
                device_map='auto', trust_remote_code=True, torch_dtype=torch.float16)
    model.eval()
    preds = []
    for msgs in PROMPTS_MSGS:
        try:
            prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ids = tok(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=256, do_sample=False, pad_token_id=tok.eos_token_id)
        preds.append(parse(tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)))
    del model; gc.collect(); torch.cuda.empty_cache()
    # 집계
    ok = [(g, p) for g, p in zip(GOLD, preds) if p is not None]
    agree = sum(1 for g, p in ok if g == p)
    from collections import Counter
    per = {}
    for c in ('yes','pending','no'):
        gc_ = [(g, p) for g, p in zip(GOLD, preds) if g == c]
        per[c] = (sum(1 for g, p in gc_ if p == c), len(gc_))
    conf = Counter((g, p) for g, p in zip(GOLD, preds) if g != p)
    return {'model': model_id, 'n_valid': len(ok), 'parse_fail': len(preds)-len(ok),
            'agree_pct': agree/max(1,len(ok))*100, 'per_class': per,
            'top_conf': conf.most_common(5), 'sec': time.time()-t0}

MODELS = ['Qwen/Qwen3-14B', 'Qwen/Qwen3-30B-A3B-Instruct-2507', 'LGAI-EXAONE/EXAONE-4.0-32B']
RESULTS = []
for mid in MODELS:
    print(f'\n━━━ 평가: {mid} ━━━')
    try:
        r = evaluate(mid); RESULTS.append(r)
        print(f"  일치율 {r['agree_pct']:.1f}% (유효 {r['n_valid']}, 파싱실패 {r['parse_fail']}, {r['sec']:.0f}s)")
        print('  클래스별:', {k:f'{v[0]}/{v[1]}' for k,v in r['per_class'].items()})
    except Exception as e:
        print('  실패:', repr(e)[:200]); RESULTS.append({'model':mid,'agree_pct':0,'error':str(e)[:120]})

## 4 비교표 + 추천

In [ ]:
print('='*70)
print(f"{'모델':40} {'일치율':>8} {'yes':>7} {'pending':>9} {'no':>7}")
print('-'*70)
for r in sorted(RESULTS, key=lambda x: -x.get('agree_pct',0)):
    if 'per_class' in r:
        pc = r['per_class']
        f = lambda c: f"{pc[c][0]}/{pc[c][1]}"
        print(f"{r['model']:40} {r['agree_pct']:7.1f}% {f('yes'):>7} {f('pending'):>9} {f('no'):>7}")
    else:
        print(f"{r['model']:40}   ERROR  {r.get('error','')[:40]}")
print('='*70)
best = max((r for r in RESULTS if 'per_class' in r), key=lambda x: x['agree_pct'], default=None)
if best:
    print(f"\n→ 최고: {best['model']}  {best['agree_pct']:.1f}%")
    print('  ≥90% → c3 labeler로 채택. 미만이면 Haiku 한도상향 또는 더 큰 모델.')
    print('  pending 일치(가장 어려운 클래스)도 함께 보라 — 낮으면 그 모델은 경계 약함.')